In [1]:
import os
import numpy as np
import pandas as pd

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
dataset_path = "/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/raw/UAH-DRIVESET-v1"

print(dataset_path)

/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/raw/UAH-DRIVESET-v1


In [6]:
def load_gps(gps_path):

    gps_columns = [

        "timestamp",
        "speed",
        "latitude",
        "longitude",
        "altitude",

        "gps_quality",
        "satellites",

        "heading",

        "extra_1",
        "extra_2",
        "extra_3",
        "extra_4"

    ]

    gps_df = pd.read_csv(
        gps_path,
        sep=r"\s+",
        header=None,
        names=gps_columns
    )

    return gps_df

def synchronize_sensors(acc_df, gps_df):

    master_df = pd.merge_asof(

        acc_df.sort_values("timestamp"),

        gps_df.sort_values("timestamp"),

        on="timestamp",

        direction="nearest"

    )

    return master_df


def load_accelerometer(acc_path):

    acc_columns = [

        "timestamp",
        "active",

        "acc_x",
        "acc_y",
        "acc_z",

        "acc_x_kf",
        "acc_y_kf",
        "acc_z_kf",

        "roll",
        "pitch",
        "yaw"

    ]

    acc_df = pd.read_csv(
        acc_path,
        sep=r"\s+",
        header=None,
        names=acc_columns
    )

    return acc_df

def merge_lane_data(master_df, lane_df):

    master_df = master_df.copy()
    lane_df = lane_df.copy()

    master_df = master_df.sort_values(
        "timestamp"
    )

    lane_df = lane_df.sort_values(
        "time"
    )

    merged_df = pd.merge_asof(

        master_df,

        lane_df,

        left_on="timestamp",

        right_on="time",

        direction="nearest"

    )

    return merged_df

def create_sliding_windows_v2(
    feature_df,
    feature_list,
    window_size
):

    all_window_features = []

    for start in range(
        0,
        len(feature_df) - window_size + 1
    ):

        end = start + window_size

        window = feature_df.iloc[start:end]

        window_stats = extract_window_features_v2(
            window,
            feature_list
        )

        all_window_features.append(
            window_stats
        )

    return pd.DataFrame(
        all_window_features
    )
def engineer_features_v2(master_df):

    master_df = master_df.copy()

    # -------------------------------------------------
    # Acceleration Features
    # -------------------------------------------------

    master_df["acc_resultant"] = np.sqrt(
        master_df["acc_x"]**2 +
        master_df["acc_y"]**2 +
        master_df["acc_z"]**2
    )

    master_df["acc_horizontal"] = np.sqrt(
        master_df["acc_x"]**2 +
        master_df["acc_y"]**2
    )

    master_df["acc_vertical"] = master_df["acc_z"]

    # -------------------------------------------------
    # Delta Features
    # -------------------------------------------------

    master_df["speed_delta"] = master_df["speed"].diff().fillna(0)

    master_df["heading_delta"] = master_df["heading"].diff().fillna(0)

    master_df["roll_delta"] = master_df["roll"].diff().fillna(0)

    master_df["pitch_delta"] = master_df["pitch"].diff().fillna(0)

    master_df["yaw_delta"] = master_df["yaw"].diff().fillna(0)

    return master_df

def clean_lane_detection(lane_df):

    lane_df = lane_df.copy()

    lane_df.replace(-9, np.nan, inplace=True)

    lane_df.replace(-99, np.nan, inplace=True)

    lane_df = lane_df.sort_values(
        "time"
    )

    lane_df = lane_df.reset_index(
        drop=True
    )

    return lane_df

def load_lane_detection(lane_path):

    lane_df = pd.read_csv(
        lane_path,
        sep=r"\s+",
        header=None
    )

    lane_df.columns = [

        "time",

        "lane_offset",

        "phi",

        "road_width",

        "lane_state"
    ]

    return lane_df

def parse_trip_info(trip_name):

    parts = trip_name.split("-")

    return {
        "date": parts[0],
        "distance": parts[1],
        "driver": parts[2],
        "behavior": parts[3],
        "road_type": parts[4]
    }

def simplify_behavior(label):

     if "NORMAL" in label:
        return "NORMAL"

     if "AGGRESSIVE" in label:
        return "AGGRESSIVE"

     if "DROWSY" in label:
        return "DROWSY"

     return label




In [4]:
def process_trip_v4(
    dataset_path,
    driver,
    trip,
    window_size
):

    # --------------------------------------------------
    # Paths
    # --------------------------------------------------

    trip_path = os.path.join(
        dataset_path,
        driver,
        trip
    )

    acc_path = os.path.join(
        trip_path,
        "RAW_ACCELEROMETERS.txt"
    )

    gps_path = os.path.join(
        trip_path,
        "RAW_GPS.txt"
    )

    lane_path = os.path.join(
        trip_path,
        "PROC_LANE_DETECTION.txt"
    )

    # --------------------------------------------------
    # Load
    # --------------------------------------------------

    acc_df = load_accelerometer(acc_path)

    gps_df = load_gps(gps_path)

    lane_df = load_lane_detection(lane_path)

    lane_df = clean_lane_detection(
        lane_df
    )

    # --------------------------------------------------
    # Synchronize Sensors
    # --------------------------------------------------

    master_df = synchronize_sensors(
        acc_df,
        gps_df
    )

    # --------------------------------------------------
    # Merge Lane Detection
    # --------------------------------------------------

    master_df = merge_lane_data(
        master_df,
        lane_df
    )

    # İstersen "time" sütununu burada da silebiliriz
    if "time" in master_df.columns:
        master_df = master_df.drop(columns=["time"])

    # --------------------------------------------------
    # Feature Engineering
    # --------------------------------------------------

    feature_df = engineer_features_v2(
        master_df
    )

    # --------------------------------------------------
    # Sliding Window
    # --------------------------------------------------

    window_dataset = create_sliding_windows_v2(
        feature_df,
        window_features_v2,
        window_size
    )

    # --------------------------------------------------
    # Labels
    # --------------------------------------------------

    info = parse_trip_info(trip)

    window_dataset["driver"] = info["driver"]
    window_dataset["trip"] = trip
    window_dataset["road_type"] = info["road_type"]
    window_dataset["behavior"] = simplify_behavior(
        info["behavior"]
    )

    return window_dataset

In [7]:
window_features_v2 = [

    "acc_resultant",
    "acc_horizontal",

    "speed",
    "speed_delta",

    "roll",
    "pitch",
    "yaw",

    # Yeni Lane Features
    "lane_offset",
    "phi"

]

print(window_features_v2)

['acc_resultant', 'acc_horizontal', 'speed', 'speed_delta', 'roll', 'pitch', 'yaw', 'lane_offset', 'phi']


In [9]:
def create_sliding_windows_v2(
    feature_df,
    feature_list,
    window_size
):

    all_window_features = []

    for start in range(
        0,
        len(feature_df) - window_size + 1
    ):

        end = start + window_size

        window = feature_df.iloc[start:end]

        window_stats = extract_window_features_v2(
            window,
            feature_list
        )

        all_window_features.append(
            window_stats
        )

    return pd.DataFrame(
        all_window_features
    )
def extract_window_features_v2(window, feature_list):

    window_stats = {}

    for feature in feature_list:

        stats = extract_statistics_v2(window[feature])

        for stat_name, stat_value in stats.items():

            column_name = f"{feature}_{stat_name}"

            window_stats[column_name] = stat_value

    return window_stats

def extract_statistics_v2(signal):

    features = {}

    # ------------------------------
    # Basic Statistics
    # ------------------------------

    features["mean"] = signal.mean()
    features["std"] = signal.std()
    features["variance"] = signal.var()

    features["min"] = signal.min()
    features["max"] = signal.max()

    features["median"] = signal.median()

    features["rms"] = np.sqrt(
        np.mean(signal ** 2)
    )

    # ------------------------------
    # Distribution Shape
    # ------------------------------

    features["skewness"] = signal.skew()

    features["kurtosis"] = signal.kurt()

    # ------------------------------
    # Percentiles
    # ------------------------------

    features["q25"] = signal.quantile(0.25)

    features["q75"] = signal.quantile(0.75)

    features["iqr"] = (
        features["q75"] -
        features["q25"]
    )

    return features

In [10]:
sample_driver = "D1"

sample_trip = "20151110175712-16km-D1-NORMAL1-SECONDARY"

sample_dataset = process_trip_v4(
    dataset_path,
    sample_driver,
    sample_trip,
    60
)

print(sample_dataset.shape)

sample_dataset.head()

(6111, 112)


,acc_resultant_mean,acc_resultant_std,acc_resultant_variance,acc_resultant_min,acc_resultant_max,acc_resultant_median,acc_resultant_rms,acc_resultant_skewness,acc_resultant_kurtosis,acc_resultant_q25,...,phi_rms,phi_skewness,phi_kurtosis,phi_q25,phi_q75,phi_iqr,driver,trip,road_type,behavior
0,0.052610,0.022257,0.000495,0.019339,0.118106,0.048171,0.057052,0.794801,0.456685,0.034369,...,0.029030,1.578063,2.825057,-0.0,0.026,0.026,D1,20151110175712-16km-D1-NORMAL1-SECONDARY,SECONDARY,NORMAL
1,0.052771,0.022104,0.000489,0.019339,0.118106,0.048171,0.057142,0.809346,0.504046,0.035270,...,0.029095,1.562977,2.819787,-0.0,0.026,0.026,D1,20151110175712-16km-D1-NORMAL1-SECONDARY,SECONDARY,NORMAL
2,0.052418,0.022313,0.000498,0.019339,0.118106,0.046448,0.056897,0.814978,0.446186,0.034369,...,0.029095,1.562507,2.815531,-0.0,0.026,0.026,D1,20151110175712-16km-D1-NORMAL1-SECONDARY,SECONDARY,NORMAL
3,0.052048,0.022357,0.000500,0.019339,0.118106,0.045901,0.056573,0.856475,0.479302,0.034369,...,0.029160,1.531093,2.711391,-0.0,0.026,0.026,D1,20151110175712-16km-D1-NORMAL1-SECONDARY,SECONDARY,NORMAL
4,0.052207,0.022260,0.000496,0.019339,0.118106,0.045901,0.056681,0.853881,0.508790,0.035270,...,0.029162,1.529433,2.698166,-0.0,0.026,0.026,D1,20151110175712-16km-D1-NORMAL1-SECONDARY,SECONDARY,NORMAL


In [11]:
sample_dataset.columns[-10:]

Index(['phi_rms', 'phi_skewness', 'phi_kurtosis', 'phi_q25', 'phi_q75',
       'phi_iqr', 'driver', 'trip', 'road_type', 'behavior'],
      dtype='object')

In [16]:
trip_list = []

for driver in sorted(os.listdir(dataset_path)):

    if not driver.startswith("D"):
        continue

    driver_path = os.path.join(
        dataset_path,
        driver
    )

    if not os.path.isdir(driver_path):
        continue

    trips = []

    for trip in sorted(os.listdir(driver_path)):

        if os.path.isdir(
            os.path.join(driver_path, trip)
        ):
            trips.append(trip)

    for trip in trips:

        trip_list.append({

            "driver": driver,

            "trip": trip

        })

print(len(trip_list))

40


In [17]:
train_trips, test_trips = train_test_split(
    trip_list,
    test_size=0.20,
    random_state=42
)

print(len(train_trips))
print(len(test_trips))

32
8


In [18]:
train_datasets = []

for trip in train_trips:

    print(
        f"Processing Train: "
        f"{trip['driver']} - {trip['trip']}"
    )

    trip_dataset = process_trip_v4(
        dataset_path,
        trip["driver"],
        trip["trip"],
        60
    )

    train_datasets.append(trip_dataset)

Processing Train: D6 - 20151221120051-26km-D6-AGGRESSIVE-MOTORWAY
Processing Train: D1 - 20151111135612-13km-D1-DROWSY-SECONDARY
Processing Train: D4 - 20151204152848-25km-D4-NORMAL-MOTORWAY
Processing Train: D2 - 20151120135152-25km-D2-DROWSY-MOTORWAY
Processing Train: D2 - 20151120164606-16km-D2-DROWSY-SECONDARY
Processing Train: D5 - 20151211162829-16km-D5-NORMAL1-SECONDARY
Processing Train: D5 - 20151211170502-16km-D5-DROWSY-SECONDARY
Processing Train: D2 - 20151120133502-26km-D2-AGGRESSIVE-MOTORWAY
Processing Train: D3 - 20151126125458-16km-D3-NORMAL2-SECONDARY
Processing Train: D4 - 20151203175637-17km-D4-DROWSY-SECONDARY
Processing Train: D1 - 20151110175712-16km-D1-NORMAL1-SECONDARY
Processing Train: D5 - 20151211165606-12km-D5-AGGRESSIVE-SECONDARY
Processing Train: D1 - 20151111134545-16km-D1-AGGRESSIVE-SECONDARY
Processing Train: D2 - 20151120162105-17km-D2-NORMAL2-SECONDARY
Processing Train: D1 - 20151110180824-16km-D1-NORMAL2-SECONDARY
Processing Train: D5 - 20151209153137-

In [19]:
train_dataset = pd.concat(
    train_datasets,
    ignore_index=True
)

print(train_dataset.shape)

train_dataset.head()

(242965, 112)


,acc_resultant_mean,acc_resultant_std,acc_resultant_variance,acc_resultant_min,acc_resultant_max,acc_resultant_median,acc_resultant_rms,acc_resultant_skewness,acc_resultant_kurtosis,acc_resultant_q25,...,phi_rms,phi_skewness,phi_kurtosis,phi_q25,phi_q75,phi_iqr,driver,trip,road_type,behavior
0,0.062950,0.037494,0.001406,0.012961,0.178804,0.058821,0.073110,0.994585,0.711390,0.033116,...,0.061595,-1.419228,1.479126,-0.05900,0.0,0.05900,D6,20151221120051-26km-D6-AGGRESSIVE-MOTORWAY,MOTORWAY,AGGRESSIVE
1,0.060696,0.034344,0.001179,0.012961,0.144686,0.056502,0.069598,0.821860,0.113281,0.033116,...,0.063269,-1.310150,1.095994,-0.05925,0.0,0.05925,D6,20151221120051-26km-D6-AGGRESSIVE-MOTORWAY,MOTORWAY,AGGRESSIVE
2,0.060563,0.034495,0.001190,0.012961,0.144686,0.056502,0.069556,0.807773,0.093973,0.033116,...,0.063857,-1.258970,1.000306,-0.06125,0.0,0.06125,D6,20151221120051-26km-D6-AGGRESSIVE-MOTORWAY,MOTORWAY,AGGRESSIVE
3,0.059966,0.034357,0.001180,0.012961,0.144686,0.053694,0.068969,0.863860,0.203253,0.033116,...,0.063857,-1.258970,1.000306,-0.06125,0.0,0.06125,D6,20151221120051-26km-D6-AGGRESSIVE-MOTORWAY,MOTORWAY,AGGRESSIVE
4,0.059171,0.033793,0.001142,0.012961,0.144686,0.053694,0.068001,0.933059,0.452223,0.033116,...,0.063857,-1.258970,1.000306,-0.06125,0.0,0.06125,D6,20151221120051-26km-D6-AGGRESSIVE-MOTORWAY,MOTORWAY,AGGRESSIVE


In [20]:
test_datasets = []

for trip in test_trips:

    print(
        f"Processing Test: "
        f"{trip['driver']} - {trip['trip']}"
    )

    trip_dataset = process_trip_v4(
        dataset_path,
        trip["driver"],
        trip["trip"],
        60
    )

    test_datasets.append(trip_dataset)

Processing Test: D3 - 20151126132013-17km-D3-DROWSY-SECONDARY
Processing Test: D3 - 20151126124208-16km-D3-NORMAL1-SECONDARY
Processing Test: D3 - 20151126113754-26km-D3-DROWSY-MOTORWAY
Processing Test: D4 - 20151204154908-25km-D4-AGGRESSIVE-MOTORWAY
Processing Test: D1 - 20151111132348-25km-D1-DROWSY-MOTORWAY
Processing Test: D2 - 20151120163350-16km-D2-AGGRESSIVE-SECONDARY
Processing Test: D6 - 20151221112434-17km-D6-NORMAL-SECONDARY
Processing Test: D4 - 20151204160823-25km-D4-DROWSY-MOTORWAY


In [21]:
test_dataset = pd.concat(
    test_datasets,
    ignore_index=True
)

print(test_dataset.shape)

test_dataset.head()

(66060, 112)


,acc_resultant_mean,acc_resultant_std,acc_resultant_variance,acc_resultant_min,acc_resultant_max,acc_resultant_median,acc_resultant_rms,acc_resultant_skewness,acc_resultant_kurtosis,acc_resultant_q25,...,phi_rms,phi_skewness,phi_kurtosis,phi_q25,phi_q75,phi_iqr,driver,trip,road_type,behavior
0,0.069472,0.041229,0.001700,0.009487,0.146826,0.056582,0.080609,0.716307,-0.647953,0.040467,...,0.0,0.0,0.0,-0.0,0.0,0.0,D3,20151126132013-17km-D3-DROWSY-SECONDARY,SECONDARY,DROWSY
1,0.068400,0.041239,0.001701,0.009487,0.146826,0.055759,0.079692,0.779817,-0.566485,0.039345,...,0.0,0.0,0.0,-0.0,0.0,0.0,D3,20151126132013-17km-D3-DROWSY-SECONDARY,SECONDARY,DROWSY
2,0.066677,0.040049,0.001604,0.009487,0.146826,0.053646,0.077608,0.849096,-0.373484,0.039345,...,0.0,0.0,0.0,-0.0,0.0,0.0,D3,20151126132013-17km-D3-DROWSY-SECONDARY,SECONDARY,DROWSY
3,0.064657,0.038979,0.001519,0.009487,0.146826,0.051527,0.075330,0.909001,-0.183656,0.037702,...,0.0,0.0,0.0,-0.0,0.0,0.0,D3,20151126132013-17km-D3-DROWSY-SECONDARY,SECONDARY,DROWSY
4,0.062385,0.038071,0.001449,0.009487,0.146826,0.050660,0.072919,0.934266,-0.004782,0.035844,...,0.0,0.0,0.0,-0.0,0.0,0.0,D3,20151126132013-17km-D3-DROWSY-SECONDARY,SECONDARY,DROWSY


In [22]:
print(train_dataset.shape)
print(test_dataset.shape)

print(train_dataset.columns.equals(test_dataset.columns))

(242965, 112)
(66060, 112)
True


In [23]:
train_dataset.to_csv(
    "/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/processed/train_dataset_v4_lane.csv",
    index=False
)

test_dataset.to_csv(
    "/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/processed/test_dataset_v4_lane.csv",
    index=False
)

print("Lane Detection dataset saved successfully!")

Lane Detection dataset saved successfully!
